**Public archival copy.** Saved cell outputs (including embedded images) and Colab-specific runtime metadata have been removed. Original code and explanatory cell content remain unchanged. Some code refers to Google Drive paths and intermediate research artifacts that are not distributed here. The Fitzpatrick17k image files and the researchers' individual access link are **not included**. See `notebooks/README.md` for access and reproducibility notes.


# Phase 4 Correction: Controlled Fair–Dark Mixing Experiments

## Thesis

**Domain-Invariant and Explainable Deep Learning for Equitable and Generalizable Skin Lesion Malignancy Classification Across Diverse Skin Tones**

---

## Purpose of This Correction Notebook

This notebook reconstructs Phase 4 using the originally intended controlled experimental design.

The original Phase 4 experiment was designed to vary only the Fair/Dark skin-tone composition while keeping:

$$
N_{\mathrm{train}} = 800
$$

and:

$$
N_{\mathrm{benign}} = 400
$$

$$
N_{\mathrm{malignant}} = 400
$$

for every training composition.

The corrected experiments are:

| Experiment | Fair | Dark | Benign | Malignant | Total |
|---|---:|---:|---:|---:|---:|
| M1_75F_25D | 600 | 200 | 400 | 400 | 800 |
| M2_50F_50D | 400 | 400 | 400 | 400 | 800 |
| M3_25F_75D | 200 | 600 | 400 | 400 | 800 |

The original Phase 4 artifacts will remain unchanged and preserved as an audit trail.

This notebook will generate an independent corrected artifact branch.

In [ ]:
# ============================================
# Section 1.1
# Import Core Libraries
# ============================================

import os
import json
import random
import math
import hashlib
import warnings

from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader
)

from torchvision import (
    transforms,
    models
)

from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    brier_score_loss
)

warnings.filterwarnings("ignore")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# ============================================
# Section 1.2
# Mount Google Drive
# ============================================

from google.colab import drive

drive.mount(
    "/content/drive"
)

print("Google Drive mounted successfully.")

In [ ]:
# ============================================
# Section 1.3
# Define Compute Device
# ============================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Current device:", DEVICE)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

else:

    print(
        "CPU mode active."
    )

    print(
        "Dataset construction and auditing can continue safely."
    )

    print(
        "GPU will only be required when corrected model training begins."
    )

# Section 2: Original and Corrected Artifact Configuration

The original Phase 4 artifacts must remain unchanged.

Therefore, this notebook uses two separate roots:

| Branch | Purpose |
|---|---|
| Original Phase 4 | Historical implementation and audit trail |
| Corrected Phase 4 | Final controlled experiment |

No original Phase 4 CSV, checkpoint, history, result table, or statistical artifact will be overwritten.

In [ ]:
# ============================================
# Section 2.1
# Define Original Phase 4 Artifact Paths
# ============================================

ARTIFACT_PATH = Path(
    "/content/drive/MyDrive/Phase_3_Artifacts"
)

DATA_PATH = (
    ARTIFACT_PATH
    / "dataset_splits"
)

ORIGINAL_PHASE4_PATH = (
    ARTIFACT_PATH
    / "Phase_4_Mixing_Experiments"
)

ORIGINAL_TRAIN_PATH = (
    ORIGINAL_PHASE4_PATH
    / "training_sets"
)

ORIGINAL_VAL_PATH = (
    ORIGINAL_PHASE4_PATH
    / "validation_sets"
)

ORIGINAL_MODEL_PATH = (
    ORIGINAL_PHASE4_PATH
    / "models"
)

ORIGINAL_REPEATED_RUN_PATH = (
    ORIGINAL_PHASE4_PATH
    / "repeated_runs"
)

ORIGINAL_STAT_PATH = (
    ORIGINAL_PHASE4_PATH
    / "final_statistical_results"
)


print("Original Phase 4 root:")
print(ORIGINAL_PHASE4_PATH)

In [ ]:
# ============================================
# Section 2.2
# Define Corrected Phase 4 Artifact Paths
# ============================================

CORRECTED_PHASE4_PATH = (
    ARTIFACT_PATH
    / "Phase_4_Mixing_Experiments_Corrected"
)

CORRECTED_TRAIN_PATH = (
    CORRECTED_PHASE4_PATH
    / "training_sets"
)

CORRECTED_VAL_PATH = (
    CORRECTED_PHASE4_PATH
    / "validation_sets"
)

CORRECTED_MODEL_PATH = (
    CORRECTED_PHASE4_PATH
    / "models"
)

CORRECTED_HISTORY_PATH = (
    CORRECTED_PHASE4_PATH
    / "histories"
)

CORRECTED_PREDICTION_PATH = (
    CORRECTED_PHASE4_PATH
    / "predictions"
)

CORRECTED_TABLE_PATH = (
    CORRECTED_PHASE4_PATH
    / "tables"
)

CORRECTED_FIGURE_PATH = (
    CORRECTED_PHASE4_PATH
    / "figures"
)

CORRECTED_STAT_PATH = (
    CORRECTED_PHASE4_PATH
    / "final_statistical_results"
)

CORRECTED_MANIFEST_PATH = (
    CORRECTED_PHASE4_PATH
    / "manifests"
)


corrected_directories = [
    CORRECTED_PHASE4_PATH,
    CORRECTED_TRAIN_PATH,
    CORRECTED_VAL_PATH,
    CORRECTED_MODEL_PATH,
    CORRECTED_HISTORY_PATH,
    CORRECTED_PREDICTION_PATH,
    CORRECTED_TABLE_PATH,
    CORRECTED_FIGURE_PATH,
    CORRECTED_STAT_PATH,
    CORRECTED_MANIFEST_PATH
]


for path in corrected_directories:
    path.mkdir(
        parents=True,
        exist_ok=True
    )


print("Corrected Phase 4 artifact structure created.")
print()
print("Root:")
print(CORRECTED_PHASE4_PATH)

In [ ]:
# ============================================
# Section 2.3
# Verify Original and Corrected Artifact Roots
# ============================================

paths_to_verify = {
    "Artifact root": ARTIFACT_PATH,
    "Dataset splits": DATA_PATH,
    "Original Phase 4": ORIGINAL_PHASE4_PATH,
    "Original training sets": ORIGINAL_TRAIN_PATH,
    "Original validation sets": ORIGINAL_VAL_PATH,
    "Original repeated runs": ORIGINAL_REPEATED_RUN_PATH,
    "Corrected Phase 4": CORRECTED_PHASE4_PATH,
    "Corrected training sets": CORRECTED_TRAIN_PATH,
    "Corrected validation sets": CORRECTED_VAL_PATH,
    "Corrected models": CORRECTED_MODEL_PATH,
    "Corrected histories": CORRECTED_HISTORY_PATH,
    "Corrected statistics": CORRECTED_STAT_PATH
}


print("=" * 90)
print("PHASE 4 CORRECTION PATH VERIFICATION")
print("=" * 90)


all_paths_found = True


for name, path in paths_to_verify.items():

    exists = path.exists()

    if not exists:
        all_paths_found = False

    status = (
        "✓ FOUND"
        if exists
        else "✗ MISSING"
    )

    print(
        f"{name:<35} {status}"
    )


print("=" * 90)


if all_paths_found:

    print(
        "✓ Original and corrected artifact structures are ready."
    )

else:

    print(
        "⚠ One or more required paths are missing."
    )

# Section 3: Reconstruct the Corrected Phase 4 Training Sets

The corrected Phase 4 datasets will be reconstructed directly from the frozen Phase 3 training pools:

- `fair_train.csv`
- `dark_train.csv`

The original Phase 4 controlled sampler explicitly enforced equal benign and malignant counts within each requested skin-tone subset.

For a requested domain sample size:

$$
n
$$

the sampler uses:

$$
n_{\mathrm{benign}}
=
n_{\mathrm{malignant}}
=
\frac{n}{2}
$$

Therefore, each final Phase 4 training dataset contains:

$$
N_{\mathrm{benign}} = 400
$$

and:

$$
N_{\mathrm{malignant}} = 400
$$

while only the Fair/Dark training composition changes.

In [ ]:
# ============================================
# Section 3.1
# Load Frozen Phase 3 Source Training Pools
# ============================================

FAIR_TRAIN_FILE = (
    DATA_PATH
    / "fair_train.csv"
)

DARK_TRAIN_FILE = (
    DATA_PATH
    / "dark_train.csv"
)


fair_train_df = pd.read_csv(
    FAIR_TRAIN_FILE
)

dark_train_df = pd.read_csv(
    DARK_TRAIN_FILE
)


print("=" * 90)
print("FROZEN PHASE 3 SOURCE TRAINING POOLS")
print("=" * 90)

print(
    "Fair training shape:",
    fair_train_df.shape
)

print(
    "Dark training shape:",
    dark_train_df.shape
)

print("\nFair target distribution:")

print(
    fair_train_df["target"]
    .value_counts()
    .sort_index()
)

print("\nDark target distribution:")

print(
    dark_train_df["target"]
    .value_counts()
    .sort_index()
)

print("\nFair skin-tone labels:")

print(
    fair_train_df["skin_tone"]
    .value_counts(dropna=False)
)

print("\nDark skin-tone labels:")

print(
    dark_train_df["skin_tone"]
    .value_counts(dropna=False)
)

In [ ]:
# ============================================
# Section 3.2
# Define Exact Domain-and-Class Controlled Sampler
# ============================================

def sample_by_domain_and_class(
    dataframe,
    n_samples,
    random_state=42
):
    """
    Sample an exact number of images while enforcing
    equal benign and malignant class counts.

    target = 0 -> benign
    target = 1 -> malignant
    """

    if n_samples % 2 != 0:

        raise ValueError(
            "n_samples must be even."
        )


    samples_per_class = (
        n_samples // 2
    )


    benign_df = dataframe[
        dataframe["target"] == 0
    ]


    malignant_df = dataframe[
        dataframe["target"] == 1
    ]


    if len(benign_df) < samples_per_class:

        raise ValueError(
            f"Not enough benign samples. "
            f"Required: {samples_per_class}, "
            f"Available: {len(benign_df)}"
        )


    if len(malignant_df) < samples_per_class:

        raise ValueError(
            f"Not enough malignant samples. "
            f"Required: {samples_per_class}, "
            f"Available: {len(malignant_df)}"
        )


    benign_sample = benign_df.sample(
        n=samples_per_class,
        random_state=random_state
    )


    malignant_sample = malignant_df.sample(
        n=samples_per_class,
        random_state=random_state
    )


    sampled_df = pd.concat(
        [
            benign_sample,
            malignant_sample
        ],
        axis=0
    )


    sampled_df = sampled_df.sample(
        frac=1.0,
        random_state=random_state
    ).reset_index(
        drop=True
    )


    return sampled_df

In [ ]:
# ============================================
# Section 3.3
# Define Corrected Controlled Mixing Function
# ============================================

def create_controlled_mixed_dataset(
    fair_df,
    dark_df,
    fair_n,
    dark_n,
    random_state=42
):

    fair_sample = sample_by_domain_and_class(
        fair_df,
        fair_n,
        random_state=random_state
    )


    dark_sample = sample_by_domain_and_class(
        dark_df,
        dark_n,
        random_state=random_state + 1
    )


    mixed_df = pd.concat(
        [
            fair_sample,
            dark_sample
        ],
        axis=0
    )


    mixed_df = mixed_df.sample(
        frac=1.0,
        random_state=random_state + 2
    ).reset_index(
        drop=True
    )


    return mixed_df

In [ ]:
# ============================================
# Section 3.4
# Define Corrected Phase 4 Mixing Configuration
# ============================================

SEED = 42

TOTAL_TRAIN_SAMPLES = 800


MIXING_CONFIGS = {

    "M1_75F_25D": {
        "fair_ratio": 0.75,
        "dark_ratio": 0.25
    },

    "M2_50F_50D": {
        "fair_ratio": 0.50,
        "dark_ratio": 0.50
    },

    "M3_25F_75D": {
        "fair_ratio": 0.25,
        "dark_ratio": 0.75
    }
}


print("=" * 90)
print("CORRECTED PHASE 4 MIXING CONFIGURATION")
print("=" * 90)


for experiment_name, config in MIXING_CONFIGS.items():

    fair_n = int(
        TOTAL_TRAIN_SAMPLES
        * config["fair_ratio"]
    )

    dark_n = int(
        TOTAL_TRAIN_SAMPLES
        * config["dark_ratio"]
    )

    print(
        f"{experiment_name}: "
        f"Fair={fair_n}, "
        f"Dark={dark_n}, "
        f"Total={fair_n + dark_n}"
    )

In [ ]:
# ============================================
# Section 3.5
# Generate Corrected Phase 4 Training Sets
# ============================================

corrected_train_sets = {}


for experiment_name, config in MIXING_CONFIGS.items():

    fair_n = int(
        TOTAL_TRAIN_SAMPLES
        * config["fair_ratio"]
    )

    dark_n = int(
        TOTAL_TRAIN_SAMPLES
        * config["dark_ratio"]
    )


    mixed_df = create_controlled_mixed_dataset(
        fair_train_df,
        dark_train_df,
        fair_n,
        dark_n,
        random_state=SEED
    )


    corrected_train_sets[
        experiment_name
    ] = mixed_df


    print(
        "\n" + "=" * 70
    )

    print(
        experiment_name
    )

    print(
        "=" * 70
    )

    print(
        "\nSkin-tone distribution:"
    )

    print(
        mixed_df["skin_tone"]
        .value_counts()
    )

    print(
        "\nDisease distribution:"
    )

    print(
        mixed_df["target"]
        .value_counts()
        .sort_index()
    )

In [ ]:
# ============================================
# Section 3.6
# Verify Corrected Controlled Training Design
# ============================================

verification_rows = []


for experiment_name, df in corrected_train_sets.items():

    fair_count = int(
        df["skin_tone"]
        .eq("Fair")
        .sum()
    )

    dark_count = int(
        df["skin_tone"]
        .eq("Dark")
        .sum()
    )

    benign_count = int(
        df["target"]
        .eq(0)
        .sum()
    )

    malignant_count = int(
        df["target"]
        .eq(1)
        .sum()
    )

    fair_benign = int(
        (
            df["skin_tone"].eq("Fair")
            &
            df["target"].eq(0)
        ).sum()
    )

    fair_malignant = int(
        (
            df["skin_tone"].eq("Fair")
            &
            df["target"].eq(1)
        ).sum()
    )

    dark_benign = int(
        (
            df["skin_tone"].eq("Dark")
            &
            df["target"].eq(0)
        ).sum()
    )

    dark_malignant = int(
        (
            df["skin_tone"].eq("Dark")
            &
            df["target"].eq(1)
        ).sum()
    )


    verification_rows.append({

        "experiment":
            experiment_name,

        "total":
            len(df),

        "fair":
            fair_count,

        "dark":
            dark_count,

        "benign":
            benign_count,

        "malignant":
            malignant_count,

        "fair_benign":
            fair_benign,

        "fair_malignant":
            fair_malignant,

        "dark_benign":
            dark_benign,

        "dark_malignant":
            dark_malignant
    })


corrected_design_df = pd.DataFrame(
    verification_rows
)


display(
    corrected_design_df
)


expected_design = {

    "M1_75F_25D": {
        "fair": 600,
        "dark": 200
    },

    "M2_50F_50D": {
        "fair": 400,
        "dark": 400
    },

    "M3_25F_75D": {
        "fair": 200,
        "dark": 600
    }
}


all_correct = True


for _, row in corrected_design_df.iterrows():

    experiment = row["experiment"]

    expected = expected_design[
        experiment
    ]


    checks = [

        row["total"] == 800,

        row["fair"] ==
            expected["fair"],

        row["dark"] ==
            expected["dark"],

        row["benign"] == 400,

        row["malignant"] == 400

    ]


    if not all(checks):

        all_correct = False


print("=" * 90)


if all_correct:

    print(
        "✓ All corrected Phase 4 training sets "
        "match the locked experimental design."
    )

else:

    print(
        "✗ Corrected dataset verification failed."
    )

In [ ]:
# ============================================
# Section 3.7
# Save Corrected Phase 4 Training Sets
# ============================================

saved_training_files = {}


for experiment_name, df in corrected_train_sets.items():

    output_file = (
        CORRECTED_TRAIN_PATH
        / f"{experiment_name}_train.csv"
    )

    df.to_csv(
        output_file,
        index=False
    )

    saved_training_files[
        experiment_name
    ] = output_file

    print(
        f"✓ Saved {experiment_name}"
    )

    print(
        f"  {output_file}"
    )

    print(
        f"  Samples: {len(df)}"
    )

    print()


print("=" * 90)
print("✓ All corrected training CSV files saved.")
print("=" * 90)

In [ ]:
# ============================================
# Section 3.8
# Reload and Verify Saved Corrected Training Sets
# ============================================

saved_verification_rows = []


for experiment_name, file_path in saved_training_files.items():

    df = pd.read_csv(
        file_path
    )


    record = {

        "experiment":
            experiment_name,

        "total":
            len(df),

        "fair":
            int(
                df["skin_tone"]
                .eq("Fair")
                .sum()
            ),

        "dark":
            int(
                df["skin_tone"]
                .eq("Dark")
                .sum()
            ),

        "benign":
            int(
                df["target"]
                .eq(0)
                .sum()
            ),

        "malignant":
            int(
                df["target"]
                .eq(1)
                .sum()
            ),

        "duplicate_md5":
            int(
                df["md5hash"]
                .duplicated()
                .sum()
            ),

        "duplicate_path":
            int(
                df["image_path"]
                .duplicated()
                .sum()
            ),

        "missing_critical":
            int(
                df[
                    [
                        "md5hash",
                        "image_path",
                        "skin_tone",
                        "target"
                    ]
                ]
                .isna()
                .sum()
                .sum()
            )
    }


    saved_verification_rows.append(
        record
    )


saved_corrected_audit_df = pd.DataFrame(
    saved_verification_rows
)


display(
    saved_corrected_audit_df
)


all_saved_correctly = (

    (saved_corrected_audit_df["total"] == 800).all()

    and

    (saved_corrected_audit_df["benign"] == 400).all()

    and

    (saved_corrected_audit_df["malignant"] == 400).all()

    and

    (saved_corrected_audit_df["duplicate_md5"] == 0).all()

    and

    (saved_corrected_audit_df["duplicate_path"] == 0).all()

    and

    (saved_corrected_audit_df["missing_critical"] == 0).all()
)


print("=" * 90)


if all_saved_correctly:

    print(
        "✓ Saved corrected training artifacts passed integrity verification."
    )

else:

    print(
        "✗ Saved corrected training artifact verification failed."
    )

In [ ]:
# ============================================
# Section 3.9
# Compare Original and Corrected Training Artifacts
# ============================================

comparison_rows = []


for experiment_name in MIXING_CONFIGS.keys():

    original_file = (
        ORIGINAL_TRAIN_PATH
        / f"{experiment_name}_train.csv"
    )

    corrected_file = (
        CORRECTED_TRAIN_PATH
        / f"{experiment_name}_train.csv"
    )


    original_df = pd.read_csv(
        original_file
    )

    corrected_df = pd.read_csv(
        corrected_file
    )


    original_md5 = set(
        original_df["md5hash"]
    )

    corrected_md5 = set(
        corrected_df["md5hash"]
    )


    shared = (
        original_md5
        & corrected_md5
    )

    original_only = (
        original_md5
        - corrected_md5
    )

    corrected_only = (
        corrected_md5
        - original_md5
    )


    comparison_rows.append({

        "experiment":
            experiment_name,

        "original_n":
            len(original_df),

        "corrected_n":
            len(corrected_df),

        "shared_images":
            len(shared),

        "original_only":
            len(original_only),

        "corrected_only":
            len(corrected_only),

        "original_benign":
            int(
                original_df["target"]
                .eq(0)
                .sum()
            ),

        "original_malignant":
            int(
                original_df["target"]
                .eq(1)
                .sum()
            ),

        "corrected_benign":
            int(
                corrected_df["target"]
                .eq(0)
                .sum()
            ),

        "corrected_malignant":
            int(
                corrected_df["target"]
                .eq(1)
                .sum()
            )
    })


original_corrected_comparison_df = pd.DataFrame(
    comparison_rows
)


display(
    original_corrected_comparison_df
)

## Section 3.10: Corrected Training Dataset Freeze

The corrected Phase 4 training datasets are now treated as frozen.

No further resampling or modification of these CSV files will occur during corrected Phase 4.

All subsequent corrected Phase 4 models will use these exact saved training artifacts.

The original Phase 4 training CSVs remain preserved separately and are not overwritten.

This separation maintains a complete audit trail between:

$$
\text{Original Phase 4 Implementation}
$$

and:

$$
\text{Corrected Phase 4 Confirmatory Implementation}
$$

# Section 4: Lock Frozen Validation and Test Sets

The Phase 4 correction affects only the training-set construction.

The following artifacts remain unchanged:

- Fair validation split
- Dark validation split
- Fair test split
- Dark test split

These frozen evaluation datasets were inherited from Phase 3 and must remain identical so that the corrected Phase 4 models are evaluated under the same conditions as the original Phase 4 models.

The corrected experiment therefore changes only:

$$
\text{Training-set construction}
$$

while keeping:

$$
\text{Validation data}
$$

and:

$$
\text{Test data}
$$

fixed.

In [ ]:
# ============================================
# Section 4.1
# Load Frozen Phase 3 Validation and Test Splits
# ============================================

FAIR_VAL_FILE = (
    DATA_PATH
    / "fair_val.csv"
)

DARK_VAL_FILE = (
    DATA_PATH
    / "dark_val.csv"
)

FAIR_TEST_FILE = (
    DATA_PATH
    / "fair_test.csv"
)

DARK_TEST_FILE = (
    DATA_PATH
    / "dark_test.csv"
)


fair_val_df = pd.read_csv(
    FAIR_VAL_FILE
)

dark_val_df = pd.read_csv(
    DARK_VAL_FILE
)

fair_test_df = pd.read_csv(
    FAIR_TEST_FILE
)

dark_test_df = pd.read_csv(
    DARK_TEST_FILE
)


print("=" * 90)
print("FROZEN VALIDATION AND TEST SPLITS")
print("=" * 90)

print(
    "Fair validation:",
    fair_val_df.shape
)

print(
    "Dark validation:",
    dark_val_df.shape
)

print(
    "Fair test:",
    fair_test_df.shape
)

print(
    "Dark test:",
    dark_test_df.shape
)

In [ ]:
# ============================================
# Section 4.2
# Audit Frozen Validation and Test Splits
# ============================================

evaluation_splits = {

    "Fair Validation":
        fair_val_df,

    "Dark Validation":
        dark_val_df,

    "Fair Test":
        fair_test_df,

    "Dark Test":
        dark_test_df
}


audit_rows = []


for split_name, df in evaluation_splits.items():

    audit_rows.append({

        "split":
            split_name,

        "total":
            len(df),

        "benign":
            int(
                df["target"]
                .eq(0)
                .sum()
            ),

        "malignant":
            int(
                df["target"]
                .eq(1)
                .sum()
            ),

        "fair":
            int(
                df["skin_tone"]
                .eq("Fair")
                .sum()
            ),

        "dark":
            int(
                df["skin_tone"]
                .eq("Dark")
                .sum()
            ),

        "duplicate_md5":
            int(
                df["md5hash"]
                .duplicated()
                .sum()
            ),

        "missing_critical":
            int(
                df[
                    [
                        "md5hash",
                        "image_path",
                        "skin_tone",
                        "target"
                    ]
                ]
                .isna()
                .sum()
                .sum()
            )
    })


evaluation_audit_df = pd.DataFrame(
    audit_rows
)


display(
    evaluation_audit_df
)

In [ ]:
# ============================================
# Section 4.3
# Verify Corrected Training vs Evaluation Independence
# ============================================

evaluation_md5 = set(
    pd.concat(
        [
            fair_val_df,
            dark_val_df,
            fair_test_df,
            dark_test_df
        ],
        axis=0
    )["md5hash"]
)


leakage_rows = []


for experiment_name, train_df in corrected_train_sets.items():

    train_md5 = set(
        train_df["md5hash"]
    )

    overlap = (
        train_md5
        .intersection(
            evaluation_md5
        )
    )


    leakage_rows.append({

        "experiment":
            experiment_name,

        "train_samples":
            len(train_df),

        "evaluation_overlap":
            len(overlap)
    })


leakage_audit_df = pd.DataFrame(
    leakage_rows
)


display(
    leakage_audit_df
)


if (
    leakage_audit_df[
        "evaluation_overlap"
    ]
    .eq(0)
    .all()
):

    print(
        "✓ No corrected training image overlaps "
        "with validation or test data."
    )

else:

    print(
        "✗ Leakage detected."
    )

In [ ]:
# ============================================
# Section 4.4
# Verify Expected Frozen Test Composition
# ============================================

print("=" * 90)
print("FAIR TEST")
print("=" * 90)

print(
    fair_test_df["target"]
    .value_counts()
    .sort_index()
)


print("\n" + "=" * 90)
print("DARK TEST")
print("=" * 90)

print(
    dark_test_df["target"]
    .value_counts()
    .sort_index()
)


fair_test_ok = (
    len(fair_test_df) == 487
    and
    int(
        fair_test_df["target"]
        .eq(0)
        .sum()
    ) == 239
    and
    int(
        fair_test_df["target"]
        .eq(1)
        .sum()
    ) == 248
)


dark_test_ok = (
    len(dark_test_df) == 162
    and
    int(
        dark_test_df["target"]
        .eq(0)
        .sum()
    ) == 86
    and
    int(
        dark_test_df["target"]
        .eq(1)
        .sum()
    ) == 76
)


print("\n" + "=" * 90)


if fair_test_ok and dark_test_ok:

    print(
        "✓ Frozen Fair and Dark test sets match the authoritative Phase 4 design."
    )

else:

    print(
        "✗ Frozen test-set composition differs from expected values."
    )

# Section 5: Reproduce the Original Phase 4 Training Pipeline

The corrected Phase 4 experiment changes only the training-set construction.

All model-training components are retained from the original Phase 4 implementation.

The model is an ImageNet-pretrained ResNet50 classifier with:

$$
f_\theta(X)
\rightarrow
\mathbb{R}^{2}
$$

and final classifier:

$$
W_{fc}
\in
\mathbb{R}^{2 \times 2048}
$$

Training uses cross-entropy loss:

$$
\mathcal{L}_{CE}
=
-\sum_{k=1}^{2}
y_k
\log
\hat{p}_k
$$

The optimizer and training configuration remain identical to the original Phase 4 experiment.

In [ ]:
# ============================================
# Section 5.1
# Load Frozen Preprocessing Configuration
# ============================================

PREPROCESSING_CONFIG_FILE = (
    ARTIFACT_PATH
    / "preprocessing_config.json"
)


with open(
    PREPROCESSING_CONFIG_FILE,
    "r"
) as f:

    preprocessing_config = json.load(f)


IMAGE_SIZE = (
    preprocessing_config[
        "image_size"
    ]
)

NORMALIZE_MEAN = (
    preprocessing_config[
        "normalization"
    ]["mean"]
)

NORMALIZE_STD = (
    preprocessing_config[
        "normalization"
    ]["std"]
)


print("=" * 80)
print("FROZEN PREPROCESSING CONFIGURATION")
print("=" * 80)

print(
    "Image size:",
    IMAGE_SIZE
)

print(
    "Normalization mean:",
    NORMALIZE_MEAN
)

print(
    "Normalization std:",
    NORMALIZE_STD
)

In [ ]:
# ============================================
# Section 5.2
# Define Phase 3/4-Compatible Image Transforms
# ============================================

train_transform = transforms.Compose([

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.RandomRotation(
        degrees=10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=NORMALIZE_MEAN,
        std=NORMALIZE_STD
    )
])


eval_transform = transforms.Compose([

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=NORMALIZE_MEAN,
        std=NORMALIZE_STD
    )
])


print(
    "✓ Training transform created."
)

print(
    "✓ Evaluation transform created."
)

In [ ]:
# ============================================
# Section 5.3
# Define Classification Dataset
# ============================================

class ClassificationDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.dataframe = (
            dataframe
            .reset_index(
                drop=True
            )
        )

        self.transform = transform


    def __len__(
        self
    ):

        return len(
            self.dataframe
        )


    def __getitem__(
        self,
        idx
    ):

        row = (
            self.dataframe
            .iloc[idx]
        )


        image = Image.open(
            row[
                "image_path"
            ]
        ).convert(
            "RGB"
        )


        if self.transform:

            image = (
                self.transform(
                    image
                )
            )


        label = torch.tensor(
            row[
                "target"
            ],
            dtype=torch.long
        )


        return (
            image,
            label
        )

In [ ]:
# ============================================
# Section 5.4
# Define Frozen Phase 4 Training Configuration
# ============================================

NUM_CLASSES = 2

BATCH_SIZE = 32

EPOCHS = 50

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

PATIENCE = 10


REPEAT_SEEDS = [
    42,
    43,
    44,
    45,
    46
]


print("=" * 80)
print("CORRECTED PHASE 4 TRAINING CONFIGURATION")
print("=" * 80)

print(
    "Image size:",
    IMAGE_SIZE
)

print(
    "Classes:",
    NUM_CLASSES
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Maximum epochs:",
    EPOCHS
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Weight decay:",
    WEIGHT_DECAY
)

print(
    "Early stopping patience:",
    PATIENCE
)

print(
    "Seeds:",
    REPEAT_SEEDS
)

print(
    "Device:",
    DEVICE
)

In [ ]:
# ============================================
# Section 5.5
# Define Reproducibility Function
# ============================================

def set_reproducible_seed(
    seed
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed(
            seed
        )

        torch.cuda.manual_seed_all(
            seed
        )


    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


    print(
        f"Random seed set to: {seed}"
    )

In [ ]:
# ============================================
# Section 5.6
# Define ResNet50 Architecture
# ============================================

def create_resnet50():

    weights = (
        models
        .ResNet50_Weights
        .DEFAULT
    )


    model = models.resnet50(
        weights=weights
    )


    num_features = (
        model.fc
        .in_features
    )


    model.fc = nn.Linear(
        num_features,
        NUM_CLASSES
    )


    model = model.to(
        DEVICE
    )


    return model

In [ ]:
# ============================================
# Section 5.7
# Define Seeded Training and Validation Loaders
# ============================================

def create_seeded_mixed_loaders(
    train_df,
    val_df,
    seed
):

    train_dataset = (
        ClassificationDataset(
            train_df,
            transform=train_transform
        )
    )


    val_dataset = (
        ClassificationDataset(
            val_df,
            transform=eval_transform
        )
    )


    generator = torch.Generator()

    generator.manual_seed(
        seed
    )


    train_loader = DataLoader(

        train_dataset,

        batch_size=BATCH_SIZE,

        shuffle=True,

        num_workers=0,

        pin_memory=True,

        generator=generator
    )


    val_loader = DataLoader(

        val_dataset,

        batch_size=BATCH_SIZE,

        shuffle=False,

        num_workers=0,

        pin_memory=True
    )


    return (
        train_loader,
        val_loader
    )

In [ ]:
# ============================================
# Section 5.8
# Create Frozen Fair and Dark Test Loaders
# ============================================

fair_test_dataset = (
    ClassificationDataset(
        fair_test_df,
        transform=eval_transform
    )
)


dark_test_dataset = (
    ClassificationDataset(
        dark_test_df,
        transform=eval_transform
    )
)


fair_test_loader = DataLoader(

    fair_test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=True
)


dark_test_loader = DataLoader(

    dark_test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=True
)


print(
    "Fair test samples:",
    len(
        fair_test_dataset
    )
)

print(
    "Dark test samples:",
    len(
        dark_test_dataset
    )
)

In [ ]:
# ============================================
# Section 5.9
# CPU-Safe Data Pipeline Sanity Check
# ============================================

test_train_loader, test_val_loader = (
    create_seeded_mixed_loaders(

        corrected_train_sets[
            "M2_50F_50D"
        ],

        pd.concat(
            [
                fair_val_df,
                dark_val_df
            ],
            axis=0
        ).reset_index(
            drop=True
        ),

        seed=42
    )
)


images, labels = next(
    iter(
        test_train_loader
    )
)


print(
    "Training batch shape:",
    images.shape
)

print(
    "Label batch shape:",
    labels.shape
)

print(
    "Labels present:",
    sorted(
        labels
        .unique()
        .tolist()
    )
)

print(
    "Benign in batch:",
    int(
        (
            labels == 0
        ).sum()
    )
)

print(
    "Malignant in batch:",
    int(
        (
            labels == 1
        ).sum()
    )
)

In [ ]:
# ============================================
# Section 5.10
# CPU-Safe Model Architecture Audit
# ============================================

test_model = create_resnet50()


print(
    "Final classifier:"
)

print(
    test_model.fc
)


total_parameters = sum(
    p.numel()
    for p in test_model.parameters()
)


trainable_parameters = sum(
    p.numel()
    for p in test_model.parameters()
    if p.requires_grad
)


print(
    "\nTotal parameters:",
    total_parameters
)

print(
    "Trainable parameters:",
    trainable_parameters
)


if (
    total_parameters
    ==
    trainable_parameters
):

    print(
        "\n✓ Full ResNet50 is trainable, matching original Phase 4."
    )

else:

    print(
        "\n⚠ Some model parameters are frozen."
    )


del test_model

In [ ]:
# ============================================
# Section 5.11
# Train One Epoch
# ============================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer
):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0


    for images, labels in loader:

        images = images.to(
            DEVICE
        )

        labels = labels.to(
            DEVICE
        )


        optimizer.zero_grad()


        outputs = model(
            images
        )


        loss = criterion(
            outputs,
            labels
        )


        loss.backward()

        optimizer.step()


        running_loss += (
            loss.item()
            * images.size(0)
        )


        predictions = torch.argmax(
            outputs,
            dim=1
        )


        correct += (
            predictions
            == labels
        ).sum().item()


        total += (
            labels.size(0)
        )


    epoch_loss = (
        running_loss
        / total
    )


    epoch_accuracy = (
        correct
        / total
    )


    return (
        epoch_loss,
        epoch_accuracy
    )

In [ ]:
# ============================================
# Section 5.12
# Validate One Epoch
# ============================================

def validate_one_epoch(
    model,
    loader,
    criterion
):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0


    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                DEVICE
            )

            labels = labels.to(
                DEVICE
            )


            outputs = model(
                images
            )


            loss = criterion(
                outputs,
                labels
            )


            running_loss += (
                loss.item()
                * images.size(0)
            )


            predictions = torch.argmax(
                outputs,
                dim=1
            )


            correct += (
                predictions
                == labels
            ).sum().item()


            total += (
                labels.size(0)
            )


    epoch_loss = (
        running_loss
        / total
    )


    epoch_accuracy = (
        correct
        / total
    )


    return (
        epoch_loss,
        epoch_accuracy
    )

In [ ]:
# ============================================
# Section 5.13
# Repeated-Run Training Function
# ============================================

import copy


def train_repeated_run(
    train_loader,
    val_loader,
    experiment_name,
    seed
):

    set_reproducible_seed(
        seed
    )


    print(
        "=" * 70
    )

    print(
        f"Experiment: {experiment_name} | "
        f"Seed: {seed}"
    )

    print(
        "=" * 70
    )


    model = create_resnet50()


    criterion = (
        nn.CrossEntropyLoss()
    )


    optimizer = (
        torch.optim.Adam(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY
        )
    )


    best_val_loss = float(
        "inf"
    )

    best_val_accuracy = None

    best_epoch = None

    best_weights = None

    patience_counter = 0

    history = []


    for epoch in range(
        EPOCHS
    ):

        train_loss, train_acc = (
            train_one_epoch(
                model,
                train_loader,
                criterion,
                optimizer
            )
        )


        val_loss, val_acc = (
            validate_one_epoch(
                model,
                val_loader,
                criterion
            )
        )


        history.append({

            "epoch":
                epoch + 1,

            "train_loss":
                train_loss,

            "train_accuracy":
                train_acc,

            "val_loss":
                val_loss,

            "val_accuracy":
                val_acc

        })


        print(

            f"Epoch "
            f"{epoch + 1:02d}/{EPOCHS} | "

            f"Train Loss: "
            f"{train_loss:.4f} | "

            f"Train Acc: "
            f"{train_acc:.4f} | "

            f"Val Loss: "
            f"{val_loss:.4f} | "

            f"Val Acc: "
            f"{val_acc:.4f}"

        )


        if (
            val_loss
            <
            best_val_loss
        ):

            best_val_loss = (
                val_loss
            )

            best_val_accuracy = (
                val_acc
            )

            best_epoch = (
                epoch + 1
            )

            best_weights = (
                copy.deepcopy(
                    model.state_dict()
                )
            )

            patience_counter = 0

        else:

            patience_counter += 1


        if (
            patience_counter
            >= PATIENCE
        ):

            print(
                f"\nEarly stopping at epoch "
                f"{epoch + 1}."
            )

            break


    model.load_state_dict(
        best_weights
    )


    history_df = pd.DataFrame(
        history
    )


    training_summary = {

        "Experiment":
            experiment_name,

        "Seed":
            seed,

        "Best_Epoch":
            best_epoch,

        "Best_Val_Loss":
            best_val_loss,

        "Best_Val_Accuracy":
            best_val_accuracy,

        "Total_Epochs_Run":
            len(
                history_df
            )

    }


    print(
        f"\nBest epoch: "
        f"{best_epoch}"
    )

    print(
        "Best validation loss: "
        f"{best_val_loss:.4f}"
    )

    print(
        "Best validation accuracy: "
        f"{best_val_accuracy:.4f}"
    )


    return (
        model,
        history_df,
        training_summary
    )

In [ ]:
# ============================================
# Section 5.14
# Evaluate Model on a Fixed Test Domain
# ============================================

def evaluate_model(
    model,
    loader
):

    model.eval()


    all_labels = []

    all_predictions = []

    all_probabilities = []


    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                DEVICE
            )


            outputs = model(
                images
            )


            probabilities = (
                torch.softmax(
                    outputs,
                    dim=1
                )
            )


            predictions = (
                torch.argmax(
                    probabilities,
                    dim=1
                )
            )


            all_labels.extend(
                labels.numpy()
            )


            all_predictions.extend(
                predictions
                .cpu()
                .numpy()
            )


            all_probabilities.extend(
                probabilities[
                    :,
                    1
                ]
                .cpu()
                .numpy()
            )


    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )


    precision = precision_score(
        all_labels,
        all_predictions,
        zero_division=0
    )


    recall = recall_score(
        all_labels,
        all_predictions,
        zero_division=0
    )


    f1 = f1_score(
        all_labels,
        all_predictions,
        zero_division=0
    )


    auc = roc_auc_score(
        all_labels,
        all_probabilities
    )


    cm = confusion_matrix(
        all_labels,
        all_predictions
    )


    tn, fp, fn, tp = (
        cm.ravel()
    )


    fpr = (
        fp
        / (fp + tn)
    )


    fnr = (
        fn
        / (fn + tp)
    )


    specificity = (
        tn
        / (tn + fp)
    )


    return {

        "accuracy":
            accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "auc":
            auc,

        "specificity":
            specificity,

        "fpr":
            fpr,

        "fnr":
            fnr,

        "confusion_matrix":
            cm,

        "labels":
            np.array(
                all_labels
            ),

        "predictions":
            np.array(
                all_predictions
            ),

        "probabilities":
            np.array(
                all_probabilities
            )
    }

In [ ]:
# ============================================
# Section 5.15
# Load Frozen Composition-Specific Validation Sets
# ============================================

corrected_val_sets = {}


for experiment_name in MIXING_CONFIGS.keys():

    val_file = (
        ORIGINAL_VAL_PATH
        / f"{experiment_name}_val.csv"
    )


    val_df = pd.read_csv(
        val_file
    )


    corrected_val_sets[
        experiment_name
    ] = val_df


    print(
        f"{experiment_name}: "
        f"{len(val_df)} validation samples"
    )


    print(
        pd.crosstab(
            val_df["skin_tone"],
            val_df["target"]
        )
    )

    print()

In [ ]:
# ============================================
# Section 5.16
# Define Corrected Repeated-Run Experiments
# ============================================

CORRECTED_EXPERIMENTS = {

    "M1_75F_25D": {

        "train_df":
            corrected_train_sets[
                "M1_75F_25D"
            ],

        "val_df":
            corrected_val_sets[
                "M1_75F_25D"
            ]
    },


    "M2_50F_50D": {

        "train_df":
            corrected_train_sets[
                "M2_50F_50D"
            ],

        "val_df":
            corrected_val_sets[
                "M2_50F_50D"
            ]
    },


    "M3_25F_75D": {

        "train_df":
            corrected_train_sets[
                "M3_25F_75D"
            ],

        "val_df":
            corrected_val_sets[
                "M3_25F_75D"
            ]
    }
}


for experiment_name, config in (
    CORRECTED_EXPERIMENTS.items()
):

    print(
        experiment_name,
        "| Train:",
        len(
            config["train_df"]
        ),
        "| Validation:",
        len(
            config["val_df"]
        )
    )

# Section 6: Corrected Phase 4 Repeated Training

> **GPU REQUIRED FROM THIS SECTION**

The corrected Phase 4 confirmatory experiment consists of:

$$
3\ \text{training compositions}
\times
5\ \text{random seeds}
=
15\ \text{independently trained models}
$$

with seeds:

$$
42,\ 43,\ 44,\ 45,\ 46
$$

Each run uses the same architecture, preprocessing, optimizer, validation procedure, early-stopping criterion, and fixed Fair/Dark test sets.

Only the corrected controlled training dataset differs from the original Phase 4 implementation.

In [ ]:
# ============================================
# Section 6.1
# GPU Training Readiness Audit
# ============================================

print(
    "Device:",
    DEVICE
)

print(
    "CUDA available:",
    torch.cuda.is_available()
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(
            0
        )
    )

    print(
        "GPU memory allocated:",
        round(
            torch.cuda.memory_allocated(0)
            / 1024**3,
            3
        ),
        "GB"
    )

    print(
        "GPU memory reserved:",
        round(
            torch.cuda.memory_reserved(0)
            / 1024**3,
            3
        ),
        "GB"
    )


    print(
        "\n✓ GPU is ready for corrected Phase 4 training."
    )

else:

    print(
        "\n✗ GPU is not available."
    )

In [ ]:
# ============================================
# Section 6.2
# Define Resume-Safe Run Artifact Paths
# ============================================

CORRECTED_RUN_RESULT_FILE = (
    CORRECTED_TABLE_PATH
    / "Phase4_Corrected_Repeated_Run_Results.csv"
)


def get_run_paths(
    experiment_name,
    seed
):

    run_name = (
        f"{experiment_name}_seed_{seed}"
    )


    checkpoint_file = (
        CORRECTED_MODEL_PATH
        / f"{run_name}.pth"
    )


    history_file = (
        CORRECTED_HISTORY_PATH
        / f"{run_name}_history.csv"
    )


    metrics_file = (
        CORRECTED_TABLE_PATH
        / f"{run_name}_metrics.csv"
    )


    return {
        "run_name":
            run_name,

        "checkpoint":
            checkpoint_file,

        "history":
            history_file,

        "metrics":
            metrics_file
    }


print(
    "Repeated-run summary file:"
)

print(
    CORRECTED_RUN_RESULT_FILE
)

In [ ]:
# ============================================
# Section 6.3
# Define Completed-Run Check
# ============================================

def corrected_run_complete(
    experiment_name,
    seed
):

    paths = get_run_paths(
        experiment_name,
        seed
    )


    required_files = [

        paths["checkpoint"],

        paths["history"],

        paths["metrics"]

    ]


    return all(
        path.exists()
        for path in required_files
    )

In [ ]:
# ============================================
# Section 6.4
# Save One Completed Corrected Run
# ============================================

def save_corrected_run(
    model,
    history_df,
    training_summary,
    fair_metrics,
    dark_metrics,
    experiment_name,
    seed
):

    paths = get_run_paths(
        experiment_name,
        seed
    )


    # ----------------------------------------
    # Save checkpoint
    # ----------------------------------------

    torch.save(
        model.state_dict(),
        paths["checkpoint"]
    )


    # ----------------------------------------
    # Save training history
    # ----------------------------------------

    history_df.to_csv(
        paths["history"],
        index=False
    )


    # ----------------------------------------
    # Construct run-level metrics
    # ----------------------------------------

    result = {

        "Experiment":
            experiment_name,

        "Seed":
            seed,

        "Best_Epoch":
            training_summary[
                "Best_Epoch"
            ],

        "Best_Val_Loss":
            training_summary[
                "Best_Val_Loss"
            ],

        "Best_Val_Accuracy":
            training_summary[
                "Best_Val_Accuracy"
            ],

        "Total_Epochs_Run":
            training_summary[
                "Total_Epochs_Run"
            ],


        # Fair test
        "Fair_Accuracy":
            fair_metrics[
                "accuracy"
            ],

        "Fair_Precision":
            fair_metrics[
                "precision"
            ],

        "Fair_Recall":
            fair_metrics[
                "recall"
            ],

        "Fair_F1":
            fair_metrics[
                "f1"
            ],

        "Fair_AUC":
            fair_metrics[
                "auc"
            ],

        "Fair_Specificity":
            fair_metrics[
                "specificity"
            ],

        "Fair_FPR":
            fair_metrics[
                "fpr"
            ],

        "Fair_FNR":
            fair_metrics[
                "fnr"
            ],


        # Dark test
        "Dark_Accuracy":
            dark_metrics[
                "accuracy"
            ],

        "Dark_Precision":
            dark_metrics[
                "precision"
            ],

        "Dark_Recall":
            dark_metrics[
                "recall"
            ],

        "Dark_F1":
            dark_metrics[
                "f1"
            ],

        "Dark_AUC":
            dark_metrics[
                "auc"
            ],

        "Dark_Specificity":
            dark_metrics[
                "specificity"
            ],

        "Dark_FPR":
            dark_metrics[
                "fpr"
            ],

        "Dark_FNR":
            dark_metrics[
                "fnr"
            ],

        # Domain disparities
        "Accuracy_Gap":
            abs(
                fair_metrics["accuracy"]
                -
                dark_metrics["accuracy"]
            ),

        "Recall_Gap":
            abs(
                fair_metrics["recall"]
                -
                dark_metrics["recall"]
            ),

        "F1_Gap":
            abs(
                fair_metrics["f1"]
                -
                dark_metrics["f1"]
            ),

        "AUC_Gap":
            abs(
                fair_metrics["auc"]
                -
                dark_metrics["auc"]
            ),

        "FNR_Gap":
            abs(
                fair_metrics["fnr"]
                -
                dark_metrics["fnr"]
            )
    }


    run_metrics_df = pd.DataFrame(
        [result]
    )


    run_metrics_df.to_csv(
        paths["metrics"],
        index=False
    )


    return result

In [ ]:
# ============================================
# Section 6.5
# Rebuild Master Results Table from Saved Runs
# ============================================

def rebuild_corrected_results_table():

    rows = []


    for experiment_name in (
        CORRECTED_EXPERIMENTS.keys()
    ):

        for seed in REPEAT_SEEDS:

            paths = get_run_paths(
                experiment_name,
                seed
            )


            if paths["metrics"].exists():

                run_df = pd.read_csv(
                    paths["metrics"]
                )

                rows.append(
                    run_df.iloc[0]
                    .to_dict()
                )


    results_df = pd.DataFrame(
        rows
    )


    if len(results_df) > 0:

        results_df = (
            results_df
            .sort_values(
                [
                    "Experiment",
                    "Seed"
                ]
            )
            .reset_index(
                drop=True
            )
        )


        results_df.to_csv(
            CORRECTED_RUN_RESULT_FILE,
            index=False
        )


    return results_df

In [ ]:
# ============================================
# Section 6.6
# Train 15 Corrected Phase 4 Models
# Resume-Safe Execution
# ============================================

print("=" * 90)
print("CORRECTED PHASE 4 REPEATED TRAINING")
print("=" * 90)

print(
    "Experiments:",
    list(
        CORRECTED_EXPERIMENTS.keys()
    )
)

print(
    "Seeds:",
    REPEAT_SEEDS
)

print(
    "Total planned runs:",
    len(CORRECTED_EXPERIMENTS)
    *
    len(REPEAT_SEEDS)
)

print("=" * 90)


for experiment_name, config in (
    CORRECTED_EXPERIMENTS.items()
):

    train_df = (
        config["train_df"]
    )

    val_df = (
        config["val_df"]
    )


    for seed in REPEAT_SEEDS:

        print(
            "\n" + "#" * 90
        )

        print(
            f"{experiment_name} | "
            f"Seed {seed}"
        )

        print(
            "#" * 90
        )


        # ====================================
        # Resume protection
        # ====================================

        if corrected_run_complete(
            experiment_name,
            seed
        ):

            print(
                "✓ Completed artifacts already exist."
            )

            print(
                "Skipping this run."
            )

            continue


        # ====================================
        # Seed
        # ====================================

        set_reproducible_seed(
            seed
        )


        # ====================================
        # Create loaders
        # ====================================

        train_loader, val_loader = (
            create_seeded_mixed_loaders(
                train_df,
                val_df,
                seed
            )
        )


        # ====================================
        # Train
        # ====================================

        model, history_df, training_summary = (
            train_repeated_run(
                train_loader,
                val_loader,
                experiment_name,
                seed
            )
        )


        # ====================================
        # Evaluate fixed Fair test set
        # ====================================

        fair_metrics = evaluate_model(
            model,
            fair_test_loader
        )


        # ====================================
        # Evaluate fixed Dark test set
        # ====================================

        dark_metrics = evaluate_model(
            model,
            dark_test_loader
        )


        # ====================================
        # Save immediately
        # ====================================

        result = save_corrected_run(

            model=model,

            history_df=history_df,

            training_summary=training_summary,

            fair_metrics=fair_metrics,

            dark_metrics=dark_metrics,

            experiment_name=experiment_name,

            seed=seed
        )


        # ====================================
        # Rebuild master CSV
        # ====================================

        current_results = (
            rebuild_corrected_results_table()
        )


        # ====================================
        # Display current result
        # ====================================

        print(
            "\nFair Test:"
        )

        print(
            f"AUC     = "
            f"{fair_metrics['auc']:.4f}"
        )

        print(
            f"Recall  = "
            f"{fair_metrics['recall']:.4f}"
        )

        print(
            f"Accuracy= "
            f"{fair_metrics['accuracy']:.4f}"
        )


        print(
            "\nDark Test:"
        )

        print(
            f"AUC     = "
            f"{dark_metrics['auc']:.4f}"
        )

        print(
            f"Recall  = "
            f"{dark_metrics['recall']:.4f}"
        )

        print(
            f"Accuracy= "
            f"{dark_metrics['accuracy']:.4f}"
        )


        print(
            "\nDomain gaps:"
        )

        print(
            f"AUC gap    = "
            f"{result['AUC_Gap']:.4f}"
        )

        print(
            f"Recall gap = "
            f"{result['Recall_Gap']:.4f}"
        )


        print(
            "\n✓ Run artifacts saved to Drive."
        )


        # ====================================
        # Release GPU memory
        # ====================================

        del model

        del train_loader

        del val_loader


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


print(
    "\n" + "=" * 90
)

print(
    "TRAINING LOOP FINISHED"
)

print(
    "=" * 90
)


corrected_results_df = (
    rebuild_corrected_results_table()
)


print(
    "Completed corrected runs:",
    len(
        corrected_results_df
    ),
    "/ 15"
)

# Section 7: Audit Corrected Phase 4 Repeated Runs

Before statistical analysis, all corrected repeated-run artifacts are audited.

The audit verifies that:

- all 15 runs completed,
- each experiment contains exactly five seeds,
- every checkpoint exists,
- every history file exists,
- every run-level metrics file exists,
- all saved checkpoints can be loaded into the expected ResNet50 architecture,
- no duplicate experiment-seed runs exist,
- the master results table is internally consistent.

Only after this audit passes will corrected Phase 4 statistics be computed.

In [ ]:
# ============================================
# Section 7.1
# Audit Saved Run Artifacts
# ============================================

artifact_audit_rows = []


for experiment_name in CORRECTED_EXPERIMENTS.keys():

    for seed in REPEAT_SEEDS:

        paths = get_run_paths(
            experiment_name,
            seed
        )


        artifact_audit_rows.append({

            "Experiment":
                experiment_name,

            "Seed":
                seed,

            "Checkpoint_Exists":
                paths["checkpoint"].exists(),

            "History_Exists":
                paths["history"].exists(),

            "Metrics_Exists":
                paths["metrics"].exists()
        })


artifact_audit_df = pd.DataFrame(
    artifact_audit_rows
)


display(
    artifact_audit_df
)


print("=" * 90)


all_artifacts_present = (

    artifact_audit_df[
        [
            "Checkpoint_Exists",
            "History_Exists",
            "Metrics_Exists"
        ]
    ]
    .all()
    .all()
)


if all_artifacts_present:

    print(
        "✓ All 15 corrected runs have complete saved artifacts."
    )

else:

    print(
        "✗ One or more corrected runs have missing artifacts."
    )

In [ ]:
# ============================================
# Section 7.2
# Rebuild and Audit Master Results Table
# ============================================

corrected_results_df = (
    rebuild_corrected_results_table()
)


print(
    "Master results shape:",
    corrected_results_df.shape
)


display(
    corrected_results_df
)


print("\nRuns per experiment:")

print(
    corrected_results_df[
        "Experiment"
    ]
    .value_counts()
    .sort_index()
)


print("\nSeeds per experiment:")

for experiment_name in (
    CORRECTED_EXPERIMENTS.keys()
):

    seeds_found = sorted(

        corrected_results_df.loc[
            corrected_results_df[
                "Experiment"
            ]
            ==
            experiment_name,
            "Seed"
        ]
        .astype(int)
        .tolist()
    )

    print(
        experiment_name,
        ":",
        seeds_found
    )

In [ ]:
# ============================================
# Section 7.3
# Check Duplicate Experiment-Seed Runs
# ============================================

duplicate_run_count = int(

    corrected_results_df[
        [
            "Experiment",
            "Seed"
        ]
    ]
    .duplicated()
    .sum()
)


print(
    "Duplicate experiment-seed rows:",
    duplicate_run_count
)


if duplicate_run_count == 0:

    print(
        "✓ No duplicate repeated runs detected."
    )

else:

    print(
        "✗ Duplicate repeated runs detected."
    )

In [ ]:
# ============================================
# Section 7.4
# Validate Metric Ranges and Missing Values
# ============================================

metric_columns = [

    "Best_Val_Loss",
    "Best_Val_Accuracy",

    "Fair_Accuracy",
    "Fair_Precision",
    "Fair_Recall",
    "Fair_F1",
    "Fair_AUC",
    "Fair_Specificity",
    "Fair_FPR",
    "Fair_FNR",

    "Dark_Accuracy",
    "Dark_Precision",
    "Dark_Recall",
    "Dark_F1",
    "Dark_AUC",
    "Dark_Specificity",
    "Dark_FPR",
    "Dark_FNR",

    "Accuracy_Gap",
    "Recall_Gap",
    "F1_Gap",
    "AUC_Gap",
    "FNR_Gap"
]


missing_metric_count = int(

    corrected_results_df[
        metric_columns
    ]
    .isna()
    .sum()
    .sum()
)


print(
    "Missing metric values:",
    missing_metric_count
)


probability_metric_columns = [

    col
    for col in metric_columns
    if col != "Best_Val_Loss"
]


out_of_range = {}


for col in probability_metric_columns:

    invalid = (

        (
            corrected_results_df[col]
            < 0
        )

        |

        (
            corrected_results_df[col]
            > 1
        )
    ).sum()


    out_of_range[
        col
    ] = int(
        invalid
    )


out_of_range_df = pd.DataFrame({

    "Metric":
        list(
            out_of_range.keys()
        ),

    "Out_of_Range_Count":
        list(
            out_of_range.values()
        )
})


display(
    out_of_range_df
)


if (
    missing_metric_count == 0
    and
    out_of_range_df[
        "Out_of_Range_Count"
    ]
    .eq(0)
    .all()
):

    print(
        "✓ All saved metrics passed validity checks."
    )

else:

    print(
        "✗ Metric validity issue detected."
    )

In [ ]:
# ============================================
# Section 7.5
# Verify All Saved Checkpoints Can Be Loaded
# ============================================

checkpoint_audit_rows = []


for experiment_name in (
    CORRECTED_EXPERIMENTS.keys()
):

    for seed in REPEAT_SEEDS:

        paths = get_run_paths(
            experiment_name,
            seed
        )


        try:

            audit_model = (
                create_resnet50()
            )


            state_dict = torch.load(
                paths["checkpoint"],
                map_location=DEVICE
            )


            audit_model.load_state_dict(
                state_dict
            )


            checkpoint_ok = True

            error_message = ""


        except Exception as e:

            checkpoint_ok = False

            error_message = str(e)


        checkpoint_audit_rows.append({

            "Experiment":
                experiment_name,

            "Seed":
                seed,

            "Checkpoint_Load_OK":
                checkpoint_ok,

            "Error":
                error_message
        })


        if "audit_model" in locals():

            del audit_model


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


checkpoint_audit_df = pd.DataFrame(
    checkpoint_audit_rows
)


display(
    checkpoint_audit_df
)


if checkpoint_audit_df[
    "Checkpoint_Load_OK"
].all():

    print(
        "✓ All 15 corrected checkpoints loaded successfully."
    )

else:

    print(
        "✗ One or more checkpoints failed to load."
    )

In [ ]:
# ============================================
# Section 7.6
# Audit Training Histories
# ============================================

history_audit_rows = []


for experiment_name in (
    CORRECTED_EXPERIMENTS.keys()
):

    for seed in REPEAT_SEEDS:

        paths = get_run_paths(
            experiment_name,
            seed
        )


        history_df = pd.read_csv(
            paths["history"]
        )


        history_audit_rows.append({

            "Experiment":
                experiment_name,

            "Seed":
                seed,

            "Epochs":
                len(
                    history_df
                ),

            "Min_Val_Loss":
                history_df[
                    "val_loss"
                ]
                .min(),

            "Best_Recorded_Val_Loss":
                corrected_results_df.loc[
                    (
                        corrected_results_df[
                            "Experiment"
                        ]
                        ==
                        experiment_name
                    )
                    &
                    (
                        corrected_results_df[
                            "Seed"
                        ]
                        ==
                        seed
                    ),
                    "Best_Val_Loss"
                ]
                .iloc[0]
        })


history_audit_df = pd.DataFrame(
    history_audit_rows
)


history_audit_df[
    "Loss_Difference"
] = (

    history_audit_df[
        "Min_Val_Loss"
    ]

    -

    history_audit_df[
        "Best_Recorded_Val_Loss"
    ]

).abs()


display(
    history_audit_df
)


max_loss_difference = (

    history_audit_df[
        "Loss_Difference"
    ]
    .max()
)


print(
    "Maximum history vs saved best-loss difference:",
    max_loss_difference
)


if max_loss_difference < 1e-10:

    print(
        "✓ Training histories agree with saved best validation losses."
    )

else:

    print(
        "⚠ Training-history mismatch detected."
    )

# Section 8: Corrected Phase 4 Descriptive Results

After successful artifact and checkpoint auditing, the corrected repeated-run results are summarized across the five independent random seeds for each training composition.

For each outcome, the primary descriptive statistics are:

$$
\text{Mean}
$$

and:

$$
\text{Standard Deviation}
$$

across the five seed-specific models.

The experimental unit is the independently trained seed-specific model:

$$
n=5
$$

per training composition.

Images are not treated as independent experimental replications.

In [ ]:
# ============================================
# Section 8.1
# Define Primary Corrected Phase 4 Outcomes
# ============================================

PRIMARY_METRICS = [

    "Fair_Accuracy",
    "Dark_Accuracy",

    "Fair_Recall",
    "Dark_Recall",

    "Fair_F1",
    "Dark_F1",

    "Fair_AUC",
    "Dark_AUC",

    "Accuracy_Gap",
    "Recall_Gap",
    "F1_Gap",
    "AUC_Gap",

    "Fair_FNR",
    "Dark_FNR",
    "FNR_Gap"
]


print(
    "Primary outcomes:",
    len(PRIMARY_METRICS)
)

In [ ]:
# ============================================
# Section 8.2
# Compute Mean and Standard Deviation
# ============================================

summary_rows = []


for experiment_name in (
    CORRECTED_EXPERIMENTS.keys()
):

    exp_df = (
        corrected_results_df[
            corrected_results_df[
                "Experiment"
            ]
            ==
            experiment_name
        ]
    )


    for metric in PRIMARY_METRICS:

        values = (
            exp_df[
                metric
            ]
            .astype(float)
        )


        summary_rows.append({

            "Experiment":
                experiment_name,

            "Metric":
                metric,

            "N":
                len(values),

            "Mean":
                values.mean(),

            "SD":
                values.std(
                    ddof=1
                ),

            "Min":
                values.min(),

            "Max":
                values.max()
        })


corrected_summary_long_df = (
    pd.DataFrame(
        summary_rows
    )
)


display(
    corrected_summary_long_df
)

In [ ]:
# ============================================
# Section 8.3
# Create Compact Mean ± SD Results Table
# ============================================

compact_metrics = [

    "Fair_AUC",
    "Dark_AUC",
    "AUC_Gap",

    "Fair_Recall",
    "Dark_Recall",
    "Recall_Gap",

    "Fair_Accuracy",
    "Dark_Accuracy",
    "Accuracy_Gap",

    "Fair_F1",
    "Dark_F1",
    "F1_Gap",

    "Fair_FNR",
    "Dark_FNR",
    "FNR_Gap"
]


compact_rows = []


for experiment_name in (
    CORRECTED_EXPERIMENTS.keys()
):

    row = {
        "Experiment":
            experiment_name
    }


    exp_df = (
        corrected_results_df[
            corrected_results_df[
                "Experiment"
            ]
            ==
            experiment_name
        ]
    )


    for metric in compact_metrics:

        mean_value = (
            exp_df[
                metric
            ]
            .mean()
        )

        sd_value = (
            exp_df[
                metric
            ]
            .std(
                ddof=1
            )
        )


        row[
            metric
        ] = (
            f"{mean_value:.4f} ± "
            f"{sd_value:.4f}"
        )


    compact_rows.append(
        row
    )


corrected_compact_summary_df = (
    pd.DataFrame(
        compact_rows
    )
)


display(
    corrected_compact_summary_df
)

In [ ]:
# ============================================
# Section 8.4
# AUC-Focused Publication Table
# ============================================

auc_rows = []


for experiment_name in (
    CORRECTED_EXPERIMENTS.keys()
):

    exp_df = (
        corrected_results_df[
            corrected_results_df[
                "Experiment"
            ]
            ==
            experiment_name
        ]
    )


    auc_rows.append({

        "Experiment":
            experiment_name,

        "Fair_AUC_Mean":
            exp_df[
                "Fair_AUC"
            ].mean(),

        "Fair_AUC_SD":
            exp_df[
                "Fair_AUC"
            ].std(
                ddof=1
            ),

        "Dark_AUC_Mean":
            exp_df[
                "Dark_AUC"
            ].mean(),

        "Dark_AUC_SD":
            exp_df[
                "Dark_AUC"
            ].std(
                ddof=1
            ),

        "AUC_Gap_Mean":
            exp_df[
                "AUC_Gap"
            ].mean(),

        "AUC_Gap_SD":
            exp_df[
                "AUC_Gap"
            ].std(
                ddof=1
            )
    })


corrected_auc_summary_df = (
    pd.DataFrame(
        auc_rows
    )
)


display(
    corrected_auc_summary_df
)

In [ ]:
# ============================================
# Section 8.5
# Recall-Focused Publication Table
# ============================================

recall_rows = []


for experiment_name in (
    CORRECTED_EXPERIMENTS.keys()
):

    exp_df = (
        corrected_results_df[
            corrected_results_df[
                "Experiment"
            ]
            ==
            experiment_name
        ]
    )


    recall_rows.append({

        "Experiment":
            experiment_name,

        "Fair_Recall_Mean":
            exp_df[
                "Fair_Recall"
            ].mean(),

        "Fair_Recall_SD":
            exp_df[
                "Fair_Recall"
            ].std(
                ddof=1
            ),

        "Dark_Recall_Mean":
            exp_df[
                "Dark_Recall"
            ].mean(),

        "Dark_Recall_SD":
            exp_df[
                "Dark_Recall"
            ].std(
                ddof=1
            ),

        "Recall_Gap_Mean":
            exp_df[
                "Recall_Gap"
            ].mean(),

        "Recall_Gap_SD":
            exp_df[
                "Recall_Gap"
            ].std(
                ddof=1
            )
    })


corrected_recall_summary_df = (
    pd.DataFrame(
        recall_rows
    )
)


display(
    corrected_recall_summary_df
)

In [ ]:
# ============================================
# Section 8.6
# Save Corrected Descriptive Summary Tables
# ============================================

corrected_summary_long_df.to_csv(

    CORRECTED_TABLE_PATH
    / "Phase4_Corrected_Descriptive_Summary_Long.csv",

    index=False
)


corrected_compact_summary_df.to_csv(

    CORRECTED_TABLE_PATH
    / "Phase4_Corrected_Mean_SD_Summary.csv",

    index=False
)


corrected_auc_summary_df.to_csv(

    CORRECTED_TABLE_PATH
    / "Phase4_Corrected_AUC_Summary.csv",

    index=False
)


corrected_recall_summary_df.to_csv(

    CORRECTED_TABLE_PATH
    / "Phase4_Corrected_Recall_Summary.csv",

    index=False
)


print(
    "✓ Corrected Phase 4 descriptive tables saved."
)

# Section 9: Statistical Analysis of Corrected Phase 4

The independent experimental unit is the seed-specific trained model.

For each composition:

$$
n=5
$$

matched seeds are available:

$$
42,\ 43,\ 44,\ 45,\ 46
$$

The primary confirmatory repeated-measures outcomes are kept consistent with the original Phase 4 analysis:

1. AUC disparity
2. Recall disparity
3. Accuracy disparity
4. F1 disparity
5. Dark-domain AUC

Because the same seeds are used across all three training compositions, inference uses repeated-measures procedures.

Exact permutation-based Friedman tests are used because the sample size is small.

For three conditions and five matched seeds, the complete permutation space is:

$$
(3!)^5
=
7776
$$

within-seed permutations.

Pairwise follow-up comparisons use exact paired Wilcoxon signed-rank tests where appropriate.

Holm correction is applied to control multiplicity.

In [ ]:
# ============================================
# Section 9.1
# Import Statistical Utilities
# ============================================

import itertools

from scipy.stats import (
    rankdata,
    wilcoxon
)

from statsmodels.stats.multitest import (
    multipletests
)


STAT_SEED = 42

BOOTSTRAP_ITERATIONS = 10000


print(
    "Bootstrap iterations:",
    BOOTSTRAP_ITERATIONS
)

print(
    "Exact Friedman permutation space:",
    6 ** 5
)

In [ ]:
# ============================================
# Section 9.2
# Define Seed-Level Bootstrap Confidence Interval
# ============================================

def bootstrap_mean_ci(
    values,
    n_bootstrap=10000,
    confidence=0.95,
    seed=42
):

    values = np.asarray(
        values,
        dtype=float
    )


    rng = np.random.default_rng(
        seed
    )


    n = len(
        values
    )


    bootstrap_means = np.empty(
        n_bootstrap
    )


    for i in range(
        n_bootstrap
    ):

        sample = rng.choice(
            values,
            size=n,
            replace=True
        )

        bootstrap_means[i] = (
            sample.mean()
        )


    alpha = (
        1.0
        -
        confidence
    )


    lower = np.quantile(
        bootstrap_means,
        alpha / 2
    )


    upper = np.quantile(
        bootstrap_means,
        1 - alpha / 2
    )


    return (
        lower,
        upper
    )

In [ ]:
# ============================================
# Section 9.3
# Compute Bootstrap 95% Confidence Intervals
# ============================================

BOOTSTRAP_METRICS = [

    "Fair_AUC",
    "Dark_AUC",
    "AUC_Gap",

    "Fair_Recall",
    "Dark_Recall",
    "Recall_Gap",

    "Fair_Accuracy",
    "Dark_Accuracy",
    "Accuracy_Gap",

    "Fair_F1",
    "Dark_F1",
    "F1_Gap"
]


bootstrap_rows = []


for experiment_name in (
    CORRECTED_EXPERIMENTS.keys()
):

    exp_df = (
        corrected_results_df[
            corrected_results_df[
                "Experiment"
            ]
            ==
            experiment_name
        ]
    )


    for metric in BOOTSTRAP_METRICS:

        values = (
            exp_df[
                metric
            ]
            .astype(float)
            .values
        )


        lower, upper = (
            bootstrap_mean_ci(
                values,
                n_bootstrap=BOOTSTRAP_ITERATIONS,
                seed=STAT_SEED
            )
        )


        bootstrap_rows.append({

            "Experiment":
                experiment_name,

            "Metric":
                metric,

            "Mean":
                np.mean(
                    values
                ),

            "SD":
                np.std(
                    values,
                    ddof=1
                ),

            "CI95_Lower":
                lower,

            "CI95_Upper":
                upper
        })


corrected_bootstrap_df = pd.DataFrame(
    bootstrap_rows
)


display(
    corrected_bootstrap_df
)

In [ ]:
# ============================================
# Section 9.4
# Save Bootstrap Results
# ============================================

corrected_bootstrap_df.to_csv(

    CORRECTED_STAT_PATH
    / "Phase4_Corrected_Bootstrap_Confidence_Intervals.csv",

    index=False
)


print(
    "✓ Bootstrap confidence intervals saved."
)

In [ ]:
# ============================================
# Section 9 Recovery
# Restore Saved Corrected Phase 4 Results
# No Model Retraining Required
# ============================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

import itertools
import numpy as np
import pandas as pd

from scipy.stats import (
    rankdata,
    wilcoxon
)

from statsmodels.stats.multitest import (
    multipletests
)


# --------------------------------------------
# Restore artifact paths
# --------------------------------------------

ARTIFACT_PATH = Path(
    "/content/drive/MyDrive/Phase_3_Artifacts"
)

CORRECTED_PHASE4_PATH = (
    ARTIFACT_PATH
    / "Phase_4_Mixing_Experiments_Corrected"
)

CORRECTED_TABLE_PATH = (
    CORRECTED_PHASE4_PATH
    / "tables"
)

CORRECTED_STAT_PATH = (
    CORRECTED_PHASE4_PATH
    / "final_statistical_results"
)


# --------------------------------------------
# Restore master corrected results
# --------------------------------------------

CORRECTED_RUN_RESULT_FILE = (
    CORRECTED_TABLE_PATH
    / "Phase4_Corrected_Repeated_Run_Results.csv"
)

corrected_results_df = pd.read_csv(
    CORRECTED_RUN_RESULT_FILE
)


# --------------------------------------------
# Restore experiment configuration
# --------------------------------------------

REPEAT_SEEDS = [
    42,
    43,
    44,
    45,
    46
]

EXPERIMENT_ORDER = [
    "M1_75F_25D",
    "M2_50F_50D",
    "M3_25F_75D"
]


STAT_SEED = 42

BOOTSTRAP_ITERATIONS = 10000


print("=" * 90)
print("CORRECTED PHASE 4 STATISTICAL SESSION RESTORED")
print("=" * 90)

print(
    "Master results shape:",
    corrected_results_df.shape
)

print(
    "\nRuns per experiment:"
)

print(
    corrected_results_df[
        "Experiment"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\n✓ No training required."
)

print(
    "✓ CPU is sufficient for all remaining Phase 4 statistics."
)

In [ ]:
# ============================================
# Section 9.5
# Define Exact Repeated-Measures Friedman Test
# ============================================

def friedman_statistic(
    matrix
):
    """
    matrix:
        rows = matched seeds
        columns = experimental conditions
    """

    matrix = np.asarray(
        matrix,
        dtype=float
    )


    n, k = matrix.shape


    ranked = np.apply_along_axis(
        rankdata,
        1,
        matrix
    )


    rank_sums = ranked.sum(
        axis=0
    )


    q = (

        12
        /
        (
            n
            * k
            * (k + 1)
        )

        *

        np.sum(
            rank_sums ** 2
        )

        -

        3
        * n
        * (k + 1)
    )


    return float(
        q
    )

In [ ]:
# ============================================
# Section 9.6
# Exact Within-Seed Permutation Friedman Test
# ============================================

def exact_friedman_test(
    matrix
):

    matrix = np.asarray(
        matrix,
        dtype=float
    )


    n, k = (
        matrix.shape
    )


    if k != 3:

        raise ValueError(
            "This exact implementation expects exactly 3 conditions."
        )


    observed_stat = (
        friedman_statistic(
            matrix
        )
    )


    condition_permutations = list(
        itertools.permutations(
            range(k)
        )
    )


    total_permutations = (
        len(
            condition_permutations
        )
        ** n
    )


    extreme_count = 0


    for seed_permutations in itertools.product(
        condition_permutations,
        repeat=n
    ):

        permuted_matrix = np.empty_like(
            matrix
        )


        for row_index, permutation in enumerate(
            seed_permutations
        ):

            permuted_matrix[
                row_index
            ] = matrix[
                row_index,
                list(
                    permutation
                )
            ]


        permuted_stat = (
            friedman_statistic(
                permuted_matrix
            )
        )


        if (
            permuted_stat
            >=
            observed_stat
            -
            1e-12
        ):

            extreme_count += 1


    exact_p = (
        extreme_count
        /
        total_permutations
    )


    kendalls_w = (

        observed_stat
        /
        (
            n
            * (k - 1)
        )
    )


    return {

        "Friedman_Q":
            observed_stat,

        "Exact_P":
            exact_p,

        "Kendalls_W":
            kendalls_w,

        "Permutation_Count":
            total_permutations
    }

In [ ]:
# ============================================
# Section 9.7
# Define Confirmatory Phase 4 Outcomes
# ============================================

FRIEDMAN_METRICS = [

    "AUC_Gap",

    "Recall_Gap",

    "Accuracy_Gap",

    "F1_Gap",

    "Dark_AUC"

]


EXPERIMENT_ORDER = [

    "M1_75F_25D",

    "M2_50F_50D",

    "M3_25F_75D"

]


print(
    "Confirmatory outcomes:"
)

for metric in FRIEDMAN_METRICS:

    print(
        "-",
        metric
    )

In [ ]:
# ============================================
# Section 9.8
# Run Exact Friedman Tests
# ============================================

friedman_rows = []


for metric in FRIEDMAN_METRICS:

    matrix_rows = []


    for seed in REPEAT_SEEDS:

        seed_values = []


        for experiment_name in EXPERIMENT_ORDER:

            value = (

                corrected_results_df.loc[

                    (
                        corrected_results_df[
                            "Experiment"
                        ]
                        ==
                        experiment_name
                    )

                    &

                    (
                        corrected_results_df[
                            "Seed"
                        ]
                        ==
                        seed
                    ),

                    metric

                ]
                .iloc[0]
            )


            seed_values.append(
                float(
                    value
                )
            )


        matrix_rows.append(
            seed_values
        )


    matrix = np.asarray(
        matrix_rows
    )


    result = exact_friedman_test(
        matrix
    )


    friedman_rows.append({

        "Metric":
            metric,

        "Friedman_Q":
            result[
                "Friedman_Q"
            ],

        "Exact_P":
            result[
                "Exact_P"
            ],

        "Kendalls_W":
            result[
                "Kendalls_W"
            ],

        "Permutation_Count":
            result[
                "Permutation_Count"
            ]
    })


corrected_friedman_df = pd.DataFrame(
    friedman_rows
)


display(
    corrected_friedman_df
)

In [ ]:
# ============================================
# Section 9.9
# Holm Correction Across Omnibus Outcomes
# ============================================

reject, p_holm, _, _ = (
    multipletests(

        corrected_friedman_df[
            "Exact_P"
        ].values,

        alpha=0.05,

        method="holm"
    )
)


corrected_friedman_df[
    "Holm_P"
] = p_holm


corrected_friedman_df[
    "Supported_After_Holm"
] = reject


display(
    corrected_friedman_df
)

In [ ]:
# ============================================
# Section 9.10
# Define Exact Paired Wilcoxon Function
# ============================================

def exact_paired_wilcoxon(
    x,
    y
):

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )


    differences = (
        x - y
    )


    nonzero = (
        differences != 0
    )


    x_nonzero = (
        x[
            nonzero
        ]
    )

    y_nonzero = (
        y[
            nonzero
        ]
    )


    if len(
        x_nonzero
    ) == 0:

        return (
            0.0,
            1.0,
            0
        )


    result = wilcoxon(

        x_nonzero,

        y_nonzero,

        alternative="two-sided",

        zero_method="wilcox",

        method="exact"
    )


    return (

        float(
            result.statistic
        ),

        float(
            result.pvalue
        ),

        len(
            x_nonzero
        )
    )

In [ ]:
# ============================================
# Section 9.11
# Pairwise Exact Wilcoxon Follow-Up Tests
# ============================================

PAIRWISE_COMPARISONS = [

    (
        "M1_75F_25D",
        "M2_50F_50D"
    ),

    (
        "M1_75F_25D",
        "M3_25F_75D"
    ),

    (
        "M2_50F_50D",
        "M3_25F_75D"
    )
]


posthoc_rows = []


for metric in FRIEDMAN_METRICS:

    for exp_a, exp_b in (
        PAIRWISE_COMPARISONS
    ):

        df_a = (
            corrected_results_df[
                corrected_results_df[
                    "Experiment"
                ]
                ==
                exp_a
            ]
            .sort_values(
                "Seed"
            )
        )


        df_b = (
            corrected_results_df[
                corrected_results_df[
                    "Experiment"
                ]
                ==
                exp_b
            ]
            .sort_values(
                "Seed"
            )
        )


        statistic, p_value, n_nonzero = (
            exact_paired_wilcoxon(

                df_a[
                    metric
                ].values,

                df_b[
                    metric
                ].values
            )
        )


        posthoc_rows.append({

            "Metric":
                metric,

            "Comparison":
                f"{exp_a} vs {exp_b}",

            "Wilcoxon_W":
                statistic,

            "Exact_P":
                p_value,

            "N_Nonzero":
                n_nonzero
        })


corrected_posthoc_df = pd.DataFrame(
    posthoc_rows
)


display(
    corrected_posthoc_df
)

In [ ]:
# ============================================
# Section 9.12
# Holm Correction Within Each Pairwise Family
# ============================================

corrected_posthoc_df[
    "Holm_P"
] = np.nan


corrected_posthoc_df[
    "Supported_After_Holm"
] = False


for metric in FRIEDMAN_METRICS:

    mask = (
        corrected_posthoc_df[
            "Metric"
        ]
        ==
        metric
    )


    p_values = (

        corrected_posthoc_df.loc[
            mask,
            "Exact_P"
        ]
        .values
    )


    reject, adjusted_p, _, _ = (
        multipletests(

            p_values,

            alpha=0.05,

            method="holm"
        )
    )


    corrected_posthoc_df.loc[
        mask,
        "Holm_P"
    ] = adjusted_p


    corrected_posthoc_df.loc[
        mask,
        "Supported_After_Holm"
    ] = reject


display(
    corrected_posthoc_df
)

In [ ]:
# ============================================
# Section 9.13
# Save Corrected Phase 4 Statistical Results
# ============================================

corrected_friedman_df.to_csv(

    CORRECTED_STAT_PATH
    / "Phase4_Corrected_Exact_Friedman_Results.csv",

    index=False
)


corrected_posthoc_df.to_csv(

    CORRECTED_STAT_PATH
    / "Phase4_Corrected_Pairwise_Wilcoxon_Holm.csv",

    index=False
)


print(
    "✓ Corrected Phase 4 inferential statistics saved."
)

## Section 9.14: Interpretation of Pairwise Follow-Up Tests

None of the five predefined omnibus hypotheses remained statistically supported after Holm correction.

Therefore, the pairwise Wilcoxon comparisons are retained for completeness but are interpreted as exploratory follow-up analyses rather than confirmatory evidence.

In particular, F1 disparity showed an unadjusted omnibus result of:

$$
p_{\mathrm{exact}} = 0.039352
$$

with:

$$
W = 0.64
$$

but after correction across the five predefined omnibus outcomes:

$$
p_{\mathrm{Holm}} = 0.196759
$$

Therefore, no corrected Phase 4 outcome is considered statistically supported after multiplicity adjustment.

# Section 10: Original vs Corrected Phase 4 Comparison

The original Phase 4 implementation is retained as a historical reproducibility branch.

The corrected Phase 4 implementation is treated as the final controlled confirmatory experiment.

The purpose of this section is to document how correcting the training-set construction changed the observed performance patterns.

The two branches are not pooled for inferential analysis.

The comparison is descriptive and audit-oriented.

In [ ]:
# ============================================
# Section 10.1
# Load Original Phase 4 Repeated-Run Results
# ============================================

from pathlib import Path

ARTIFACT_PATH = Path(
    "/content/drive/MyDrive/Phase_3_Artifacts"
)

ORIGINAL_PHASE4_PATH = (
    ARTIFACT_PATH
    / "Phase_4_Mixing_Experiments"
)

ORIGINAL_REPEATED_RUN_PATH = (
    ORIGINAL_PHASE4_PATH
    / "repeated_runs"
)

ORIGINAL_RESULTS_FILE = (
    ORIGINAL_REPEATED_RUN_PATH
    / "Phase4_Repeated_Run_Results.csv"
)


print(
    "Looking for original results at:"
)

print(
    ORIGINAL_RESULTS_FILE
)


original_results_df = pd.read_csv(
    ORIGINAL_RESULTS_FILE
)


print("=" * 90)
print("ORIGINAL VS CORRECTED PHASE 4")
print("=" * 90)

print(
    "Original results shape:",
    original_results_df.shape
)

print(
    "Corrected results shape:",
    corrected_results_df.shape
)


print(
    "\nOriginal runs per experiment:"
)

print(
    original_results_df[
        "Experiment"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nCorrected runs per experiment:"
)

print(
    corrected_results_df[
        "Experiment"
    ]
    .value_counts()
    .sort_index()
)

In [ ]:
# ============================================
# Section 10.2
# Audit Original and Corrected Result Columns
# ============================================

print(
    "Original columns:"
)

print(
    original_results_df.columns.tolist()
)

print(
    "\nCorrected columns:"
)

print(
    corrected_results_df.columns.tolist()
)


common_columns = sorted(
    set(
        original_results_df.columns
    )
    &
    set(
        corrected_results_df.columns
    )
)


print(
    "\nNumber of common columns:",
    len(common_columns)
)

print(
    "\nCommon columns:"
)

print(
    common_columns
)

In [ ]:
# ============================================
# Section 10.3
# Compare Original and Corrected Main Outcomes
# ============================================

comparison_metrics = [

    "Fair_AUC",
    "Dark_AUC",
    "AUC_Gap",

    "Fair_Recall",
    "Dark_Recall",
    "Recall_Gap",

    "Fair_Accuracy",
    "Dark_Accuracy",
    "Accuracy_Gap",

    "Fair_F1",
    "Dark_F1",
    "F1_Gap"
]


comparison_rows = []


for experiment_name in EXPERIMENT_ORDER:

    original_exp = (
        original_results_df[
            original_results_df[
                "Experiment"
            ]
            ==
            experiment_name
        ]
    )


    corrected_exp = (
        corrected_results_df[
            corrected_results_df[
                "Experiment"
            ]
            ==
            experiment_name
        ]
    )


    for metric in comparison_metrics:

        original_values = (
            original_exp[
                metric
            ]
            .astype(float)
        )


        corrected_values = (
            corrected_exp[
                metric
            ]
            .astype(float)
        )


        comparison_rows.append({

            "Experiment":
                experiment_name,

            "Metric":
                metric,

            "Original_Mean":
                original_values.mean(),

            "Original_SD":
                original_values.std(
                    ddof=1
                ),

            "Corrected_Mean":
                corrected_values.mean(),

            "Corrected_SD":
                corrected_values.std(
                    ddof=1
                ),

            "Corrected_Minus_Original":
                corrected_values.mean()
                -
                original_values.mean()
        })


original_corrected_comparison_df = (
    pd.DataFrame(
        comparison_rows
    )
)


display(
    original_corrected_comparison_df
)

In [ ]:
# ============================================
# Section 10.4
# Create Compact AUC Comparison Table
# ============================================

auc_comparison_df = (
    original_corrected_comparison_df[
        original_corrected_comparison_df[
            "Metric"
        ]
        .isin(
            [
                "Fair_AUC",
                "Dark_AUC",
                "AUC_Gap"
            ]
        )
    ]
    .copy()
)


display(
    auc_comparison_df
)

In [ ]:
# ============================================
# Section 10.5
# Create Compact Recall Comparison Table
# ============================================

recall_comparison_df = (
    original_corrected_comparison_df[
        original_corrected_comparison_df[
            "Metric"
        ]
        .isin(
            [
                "Fair_Recall",
                "Dark_Recall",
                "Recall_Gap"
            ]
        )
    ]
    .copy()
)


display(
    recall_comparison_df
)

In [ ]:
# ============================================
# Section 10.6
# Save Original vs Corrected Comparison Tables
# ============================================

original_corrected_comparison_df.to_csv(

    CORRECTED_TABLE_PATH
    / "Phase4_Original_vs_Corrected_All_Main_Metrics.csv",

    index=False
)


auc_comparison_df.to_csv(

    CORRECTED_TABLE_PATH
    / "Phase4_Original_vs_Corrected_AUC.csv",

    index=False
)


recall_comparison_df.to_csv(

    CORRECTED_TABLE_PATH
    / "Phase4_Original_vs_Corrected_Recall.csv",

    index=False
)


print(
    "✓ Original vs corrected Phase 4 comparison tables saved."
)

## Section 10.7: Interpretation Rule

Differences between the original and corrected Phase 4 branches are interpreted as implementation-audit findings.

The corrected branch is the authoritative Phase 4 experiment because it satisfies the predefined controlled design:

$$
N=800
$$

for every composition, with:

$$
N_{\mathrm{benign}}=400
$$

and:

$$
N_{\mathrm{malignant}}=400
$$

while only Fair/Dark representation changes.

The original results remain useful for reproducibility and audit purposes but are not used as the final confirmatory Phase 4 evidence.

# Section 11: Final Corrected Phase 4 Figures

Publication-oriented figures are generated from the corrected Phase 4 results only.

The original implementation is retained for audit purposes but is not mixed into the final confirmatory figures.

The primary visualization targets are:

1. Fair and Dark AUC across training compositions
2. AUC disparity across training compositions
3. Fair and Dark recall across training compositions
4. Recall disparity across training compositions

Each point represents the mean across five independently trained seed-specific models.

Error bars represent:

$$
\pm 1\ \text{standard deviation}
$$

In [ ]:
# ============================================
# Section 11 Recovery
# Restore Corrected Figure and Manifest Paths
# ============================================

from pathlib import Path

ARTIFACT_PATH = Path(
    "/content/drive/MyDrive/Phase_3_Artifacts"
)

CORRECTED_PHASE4_PATH = (
    ARTIFACT_PATH
    / "Phase_4_Mixing_Experiments_Corrected"
)

CORRECTED_FIGURE_PATH = (
    CORRECTED_PHASE4_PATH
    / "figures"
)

CORRECTED_MANIFEST_PATH = (
    CORRECTED_PHASE4_PATH
    / "manifests"
)

CORRECTED_TABLE_PATH = (
    CORRECTED_PHASE4_PATH
    / "tables"
)

CORRECTED_STAT_PATH = (
    CORRECTED_PHASE4_PATH
    / "final_statistical_results"
)


for path in [
    CORRECTED_FIGURE_PATH,
    CORRECTED_MANIFEST_PATH,
    CORRECTED_TABLE_PATH,
    CORRECTED_STAT_PATH
]:

    path.mkdir(
        parents=True,
        exist_ok=True
    )


print(
    "✓ Corrected Phase 4 output paths restored."
)

print(
    "Figure path:",
    CORRECTED_FIGURE_PATH
)

print(
    "Manifest path:",
    CORRECTED_MANIFEST_PATH
)

In [ ]:
# ============================================
# Section 11.1
# Import Plotting Library
# ============================================

import matplotlib.pyplot as plt


PLOT_LABELS = {
    "M1_75F_25D": "75F / 25D",
    "M2_50F_50D": "50F / 50D",
    "M3_25F_75D": "25F / 75D"
}


plot_order = [
    "M1_75F_25D",
    "M2_50F_50D",
    "M3_25F_75D"
]

In [ ]:
# ============================================
# Section 11.2
# Plot Fair and Dark AUC
# ============================================

x = np.arange(
    len(plot_order)
)

fair_means = []
fair_sds = []

dark_means = []
dark_sds = []


for experiment in plot_order:

    exp_df = corrected_results_df[
        corrected_results_df[
            "Experiment"
        ] == experiment
    ]

    fair_means.append(
        exp_df[
            "Fair_AUC"
        ].mean()
    )

    fair_sds.append(
        exp_df[
            "Fair_AUC"
        ].std(ddof=1)
    )

    dark_means.append(
        exp_df[
            "Dark_AUC"
        ].mean()
    )

    dark_sds.append(
        exp_df[
            "Dark_AUC"
        ].std(ddof=1)
    )


plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    x,
    fair_means,
    yerr=fair_sds,
    marker="o",
    capsize=4,
    label="Fair Test"
)

plt.errorbar(
    x,
    dark_means,
    yerr=dark_sds,
    marker="o",
    capsize=4,
    label="Dark Test"
)

plt.xticks(
    x,
    [
        PLOT_LABELS[e]
        for e in plot_order
    ]
)

plt.xlabel(
    "Training Composition"
)

plt.ylabel(
    "AUC"
)

plt.title(
    "Corrected Phase 4: Fair and Dark Test AUC"
)

plt.legend()

plt.tight_layout()


auc_figure_file = (
    CORRECTED_FIGURE_PATH
    / "Phase4_Corrected_Fair_Dark_AUC.png"
)

plt.savefig(
    auc_figure_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


print(
    "✓ Saved:",
    auc_figure_file
)

In [ ]:
# ============================================
# Section 11.3
# Plot AUC Disparity
# ============================================

auc_gap_means = []

auc_gap_sds = []


for experiment in plot_order:

    exp_df = corrected_results_df[
        corrected_results_df[
            "Experiment"
        ] == experiment
    ]

    auc_gap_means.append(
        exp_df[
            "AUC_Gap"
        ].mean()
    )

    auc_gap_sds.append(
        exp_df[
            "AUC_Gap"
        ].std(ddof=1)
    )


plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    x,
    auc_gap_means,
    yerr=auc_gap_sds,
    marker="o",
    capsize=4
)

plt.xticks(
    x,
    [
        PLOT_LABELS[e]
        for e in plot_order
    ]
)

plt.xlabel(
    "Training Composition"
)

plt.ylabel(
    "Absolute Fair–Dark AUC Gap"
)

plt.title(
    "Corrected Phase 4: AUC Disparity"
)

plt.tight_layout()


auc_gap_figure_file = (
    CORRECTED_FIGURE_PATH
    / "Phase4_Corrected_AUC_Gap.png"
)

plt.savefig(
    auc_gap_figure_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


print(
    "✓ Saved:",
    auc_gap_figure_file
)

In [ ]:
# ============================================
# Section 11.4
# Plot Fair and Dark Recall
# ============================================

fair_recall_means = []
fair_recall_sds = []

dark_recall_means = []
dark_recall_sds = []


for experiment in plot_order:

    exp_df = corrected_results_df[
        corrected_results_df[
            "Experiment"
        ] == experiment
    ]

    fair_recall_means.append(
        exp_df[
            "Fair_Recall"
        ].mean()
    )

    fair_recall_sds.append(
        exp_df[
            "Fair_Recall"
        ].std(ddof=1)
    )

    dark_recall_means.append(
        exp_df[
            "Dark_Recall"
        ].mean()
    )

    dark_recall_sds.append(
        exp_df[
            "Dark_Recall"
        ].std(ddof=1)
    )


plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    x,
    fair_recall_means,
    yerr=fair_recall_sds,
    marker="o",
    capsize=4,
    label="Fair Test"
)

plt.errorbar(
    x,
    dark_recall_means,
    yerr=dark_recall_sds,
    marker="o",
    capsize=4,
    label="Dark Test"
)

plt.xticks(
    x,
    [
        PLOT_LABELS[e]
        for e in plot_order
    ]
)

plt.xlabel(
    "Training Composition"
)

plt.ylabel(
    "Recall"
)

plt.title(
    "Corrected Phase 4: Fair and Dark Test Recall"
)

plt.legend()

plt.tight_layout()


recall_figure_file = (
    CORRECTED_FIGURE_PATH
    / "Phase4_Corrected_Fair_Dark_Recall.png"
)

plt.savefig(
    recall_figure_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


print(
    "✓ Saved:",
    recall_figure_file
)

In [ ]:
# ============================================
# Section 11.5
# Plot Recall Disparity
# ============================================

recall_gap_means = []

recall_gap_sds = []


for experiment in plot_order:

    exp_df = corrected_results_df[
        corrected_results_df[
            "Experiment"
        ] == experiment
    ]

    recall_gap_means.append(
        exp_df[
            "Recall_Gap"
        ].mean()
    )

    recall_gap_sds.append(
        exp_df[
            "Recall_Gap"
        ].std(ddof=1)
    )


plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    x,
    recall_gap_means,
    yerr=recall_gap_sds,
    marker="o",
    capsize=4
)

plt.xticks(
    x,
    [
        PLOT_LABELS[e]
        for e in plot_order
    ]
)

plt.xlabel(
    "Training Composition"
)

plt.ylabel(
    "Absolute Fair–Dark Recall Gap"
)

plt.title(
    "Corrected Phase 4: Recall Disparity"
)

plt.tight_layout()


recall_gap_figure_file = (
    CORRECTED_FIGURE_PATH
    / "Phase4_Corrected_Recall_Gap.png"
)

plt.savefig(
    recall_gap_figure_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


print(
    "✓ Saved:",
    recall_gap_figure_file
)

In [ ]:
# ============================================
# Section 11.6
# Create Corrected Phase 4 Completion Manifest
# ============================================

import json

print("✓ json module imported.")

completion_manifest = {

    "phase":
        "Corrected Phase 4",

    "status":
        "COMPLETE",

    "training_compositions": {
        "M1_75F_25D": {
            "Fair": 600,
            "Dark": 200,
            "Benign": 400,
            "Malignant": 400
        },

        "M2_50F_50D": {
            "Fair": 400,
            "Dark": 400,
            "Benign": 400,
            "Malignant": 400
        },

        "M3_25F_75D": {
            "Fair": 200,
            "Dark": 600,
            "Benign": 400,
            "Malignant": 400
        }
    },

    "seeds":
        REPEAT_SEEDS,

    "models_trained":
        15,

    "checkpoint_audit_passed":
        True,

    "history_audit_passed":
        True,

    "metric_audit_passed":
        True,

    "train_validation_test_overlap":
        0,

    "confirmatory_outcomes": [
        "AUC_Gap",
        "Recall_Gap",
        "Accuracy_Gap",
        "F1_Gap",
        "Dark_AUC"
    ],

    "holm_supported_outcomes":
        int(
            corrected_friedman_df[
                "Supported_After_Holm"
            ].sum()
        ),

    "final_interpretation":
        (
            "No predefined corrected Phase 4 "
            "confirmatory outcome remained "
            "statistically supported after "
            "Holm correction."
        )
}


COMPLETION_MANIFEST_FILE = (
    CORRECTED_MANIFEST_PATH
    / "Phase4_Corrected_Completion_Manifest.json"
)


with open(
    COMPLETION_MANIFEST_FILE,
    "w"
) as f:

    json.dump(
        completion_manifest,
        f,
        indent=4
    )


print(
    "✓ Completion manifest saved:"
)

print(
    COMPLETION_MANIFEST_FILE
)

# Corrected Phase 4 Final Status

## COMPLETE AND FROZEN

The corrected Phase 4 experiment successfully restored the predefined controlled training design.

All three training compositions contain:

$$
N=800
$$

with:

$$
N_{\mathrm{benign}}=400
$$

and:

$$
N_{\mathrm{malignant}}=400
$$

while Fair/Dark representation varies systematically.

Fifteen corrected ResNet50 models were trained across five matched seeds per composition.

All checkpoints, training histories, metrics, validation procedures, and dataset-independence checks passed the final audit.

Descriptive differences were observed across training compositions, including variation in Dark-domain AUC and Fair–Dark recall disparity.

However, after exact repeated-measures testing and Holm correction across the predefined confirmatory Phase 4 outcomes, no outcome remained statistically supported.

Therefore, corrected Phase 4 is now frozen and serves as the authoritative upstream experiment for the corrected Phase 5 mechanistic analysis.